In [1]:
# @title 1. 安裝 RFdiffusion All Atom (極速版)
import os
import sys
import time

start_time = time.time()
print("🚀 啟動 RFdiffusionAA 安裝程序...")

# 1. 安裝與設定基礎依賴
print("⚙️ (1/5) 正在設定基礎環境 (Torch & Apptainer)...")

# 靜默安裝 PyTorch (這步最久，使用 -q 隱藏輸出)
!pip install -q torch==2.5.0
!pip uninstall -y -q torchvision torchaudio

# 安裝 aria2c 加速神器
if not os.path.exists("/usr/bin/aria2c"):
    !apt-get update -y > /dev/null
    !apt-get install -y aria2 > /dev/null

# 2. 下載程式碼 (使用 depth=1 只抓最新版)
if not os.path.isdir("rf_diffusion_all_atom"):
    print("📂 (2/5) 下載主程式碼...")
    !git clone --depth 1 https://github.com/baker-laboratory/rf_diffusion_all_atom.git
    os.chdir("rf_diffusion_all_atom")
else:
    os.chdir("rf_diffusion_all_atom")

# 3. 安裝 Apptainer (容器執行工具)
if not os.path.exists("/usr/bin/apptainer"):
    print("📦 (3/5) 安裝容器執行工具...")
    os.environ["APPTAINER_BINDPATH"] = "/content"
    !curl -J -O -s https://raw.githubusercontent.com/NeuroDesk/neurocommand/main/googlecolab_setup.sh
    !chmod +x googlecolab_setup.sh
    !./googlecolab_setup.sh > /dev/null 2>&1 # 隱藏雜訊

# 4. 極速下載模型與容器 (核心優化)
print("⬇️ (4/5) 正在全速下載模型與容器 (這會比平常快很多)...")

downloads = {
    "rf_se3_diffusion.sif": "http://files.ipd.uw.edu/pub/RF-All-Atom/containers/rf_se3_diffusion.sif",
    "RFDiffusionAA_paper_weights.pt": "http://files.ipd.uw.edu/pub/RF-All-Atom/weights/RFDiffusionAA_paper_weights.pt"
}

for filename, url in downloads.items():
    if not os.path.exists(filename):
        print(f"   正在下載: {filename} ...")
        # -x 16 代表開 16 條線程同時抓
        !aria2c -x 16 -s 16 -k 1M -o {filename} {url} > /dev/null 2>&1
        print(f"   ✅ {filename} 完成")
    else:
        print(f"   ✅ {filename} 已存在，跳過")

# 5. 初始化 Submodules
print("🔧 (5/5) 最後設定...")
!git submodule init > /dev/null
!git submodule update > /dev/null

end = int(time.time() - start_time)
print(f"\n🎉 RFdiffusionAA 安裝完成！總耗時: {end} 秒")

🚀 啟動 RFdiffusionAA 安裝程序...
⚙️ (1/5) 正在設定基礎環境 (Torch & Apptainer)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 860.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1

In [2]:
# @title RFdiffusion 裸配體口袋生成 (含自動打包下載)
# @markdown 🔴 **輸入 PDB ID，腳本將自動生成全新結合蛋白，並直接下載結果到您的電腦**

import time
import os
import sys
import urllib.request
import re
import shutil
from google.colab import files

# ==========================================
#        第一步：環境與變數設定 & 完美破解
# ==========================================
print("🔍 正在初始化環境...")

work_dir = "/content/rf_diffusion_all_atom"
if os.getcwd() != work_dir:
    os.makedirs(work_dir, exist_ok=True)
    os.chdir(work_dir)

sif_filename = "rf_se3_diffusion.sif"
if not os.path.exists(sif_filename):
    print(f"❌ 找不到映像檔，正在下載 (約 2GB)...")
    !wget https://files.ipd.uw.edu/pub/rf_diffusion/rf_se3_diffusion.sif

# ⚡ 安全還原並破解 run_inference.py (預防萬一)
file_path = "run_inference.py"
if os.path.exists(file_path):
    os.system(f"git checkout {file_path} 2>/dev/null")
    with open(file_path, "r") as f:
        code = f.read()
    match = re.search(r'^(\s*)idealize_backbone\.rewrite\(out_unidealized, out_idealized\)', code, re.MULTILINE)
    if match:
        indent = match.group(1)
        safe_new_code = (
            f"{indent}try:\n"
            f"{indent}    idealize_backbone.rewrite(out_unidealized, out_idealized)\n"
            f"{indent}except Exception as e:\n"
            f"{indent}    print('\\n⚠️ [破解成功] 略過骨架微調，強制保留原始結果')\n"
            f"{indent}    import shutil\n"
            f"{indent}    shutil.copy(out_unidealized, out_idealized)"
        )
        code = code.replace(match.group(0), safe_new_code)
        with open(file_path, "w") as f:
            f.write(code)

# ==========================================
#        參數設定區
# ==========================================
# @markdown **1. 輸入 PDB ID**
pdb_id = "heme" # @param {type:"string"}
output_prefix = "output/denovo_binder/sample" # @param {type:"string"}

# @markdown **2. 要無中生有生成多長的蛋白質？ (預設 100 個胺基酸)**
# 關鍵：不寫入A鏈，代表模型會無視原有蛋白，只針對配體生成！
contig_settings = "['100-100']" # @param {type:"string"}

# @markdown **3. 擴散模型參數**
diffuser_T = 30 # @param {type:"slider", min:10, max:100, step:5}
num_designs = 1 # @param {type:"integer"}

# 自動下載邏輯
pdb_id = pdb_id.strip().lower()
input_dir = "input"
os.makedirs(input_dir, exist_ok=True)
input_pdb = os.path.join(input_dir, f"{pdb_id}.pdb")
cleaned_pdb = os.path.join(input_dir, f"{pdb_id}_cleaned_binder.pdb")

if not os.path.exists(input_pdb):
    print(f"\n📥 正在從 RCSB PDB 下載 {pdb_id.upper()}...")
    urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb", input_pdb)

# ==========================================
#        第二步：自動解析配體
# ==========================================
print("\n🧬 正在分析 PDB 配體...")
ignore_list = {"HOH", "WAT", "SO4", "PO4", "GOL", "EDO", "ACT", "NAG", "CL", "NA", "K", "MG", "CA", "ZN", "FMT"}
found_ligands = set()

with open(input_pdb, 'r') as f:
    for line in f:
        if line.startswith("HETATM"):
            res_name = line[17:20].strip()
            if res_name not in ignore_list:
                found_ligands.add(res_name)

if not found_ligands:
    raise ValueError(f"❌ 找不到有效配體，請確認 {pdb_id.upper()} 是否包含目標小分子！")

ligand_code = list(found_ligands)[0]
print(f"🎯 鎖定目標配體: 【 {ligand_code} 】 (原蛋白將僅作為定位錨點，不參與生成干擾)")

# ==========================================
#        第三步：清洗 PDB (保留完整結構安撫 Parser)
# ==========================================
print("\n🧹 正在清洗 PDB (剔除殘缺骨架與雜質)...")
standard_aas = {"ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS", "ILE",
                "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL"}

valid_residues = set()
with open(input_pdb, 'r') as f_in:
    for line in f_in:
        if line.startswith("ATOM"):
            res_name = line[17:20].strip()
            atom_name = line[12:16].strip()
            chain = line[21]
            res_seq = line[22:26]
            if res_name in standard_aas and atom_name == "CA":
                valid_residues.add((chain, res_seq))

with open(input_pdb, 'r') as f_in, open(cleaned_pdb, 'w') as f_out:
    for line in f_in:
        if line.startswith("ATOM"):
            chain = line[21]
            res_seq = line[22:26]
            if (chain, res_seq) in valid_residues:
                alt_loc = line[16]
                if alt_loc in [' ', 'A']:
                    clean_line = line[:16] + ' ' + line[17:26] + ' ' + line[27:]
                    f_out.write(clean_line)

        elif line.startswith("HETATM"):
            res_name = line[17:20].strip()
            if res_name == ligand_code:
                alt_loc = line[16]
                if alt_loc in [' ', 'A']:
                    clean_line = line[:16] + ' ' + line[17:26] + ' ' + line[27:]
                    f_out.write(clean_line)

# ==========================================
#        第四步：執行 RFdiffusion (De novo 結合)
# ==========================================
os.makedirs(os.path.dirname(output_prefix), exist_ok=True)
print(f"\n🚀 開始執行 De novo 結合蛋白設計 (目標: {ligand_code}, 長度: {contig_settings})...")

abs_cleaned_pdb = os.path.abspath(cleaned_pdb)

!singularity run --nv {sif_filename} -u run_inference.py \
  inference.deterministic=True \
  diffuser.T={diffuser_T} \
  inference.output_prefix={output_prefix} \
  inference.input_pdb="{abs_cleaned_pdb}" \
  contigmap.contigs="{contig_settings}" \
  inference.ligand={ligand_code} \
  inference.num_designs={num_designs} \
  inference.design_startnum=0

# ==========================================
#        第五步：自動打包與下載
# ==========================================
expected_pdb = f"{output_prefix}_0.pdb"

if os.path.exists(expected_pdb):
    print(f"\n🎉 執行成功！全新設計的蛋白質已生成。")
    print("📦 正在將結果打包並準備下載...")

    # 建立一個專屬資料夾來放這次的結果
    download_dir = "/content/rf_diffusion_all_atom/download_temp"
    os.makedirs(download_dir, exist_ok=True)

    # 把 pdb 和可能產生的 trb 檔案複製過去
    for ext in ['.pdb', '.trb']:
        src_file = f"{output_prefix}_0{ext}"
        if os.path.exists(src_file):
            shutil.copy(src_file, download_dir)

    # 打包成 ZIP
    zip_path = "/content/DeNovo_Binder_Result"
    shutil.make_archive(zip_path, 'zip', download_dir)

    print("✅ 打包完成！瀏覽器即將開始下載...")
    files.download(zip_path + '.zip')
else:
    print(f"\n❌ 執行結束，但未生成檔案，請檢查日誌。")

🔍 正在初始化環境...

🧬 正在分析 PDB 配體...
🎯 鎖定目標配體: 【 HEM 】 (原蛋白將僅作為定位錨點，不參與生成干擾)

🧹 正在清洗 PDB (剔除殘缺骨架與雜質)...

🚀 開始執行 De novo 結合蛋白設計 (目標: HEM, 長度: ['100-100'])...
/bin/sh: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
DGL backend not selected or invalid.  Assuming PyTorch for now.
Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)
[05:35:49] /opt/dgl/src/runtime/tensordispatch.cc:43: TensorDispatcher: dlopen failed: /opt/conda/envs/env/lib/python3.9/site-packages/dgl/tensoradapter/pytorch/libtensoradapter_pytorch_1.13.1.so: cannot open shared object file: No such file or directory
Using backend: pytorch
/opt/conda/envs/env/lib/python3.9/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'aa': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more inform

In [3]:
# @title RFdiffusion 裸配體口袋生成 (含自動打包下載 - Dummy AA 修正版)
# @markdown 🔴 **輸入 PDB ID，腳本將自動生成全新結合蛋白，並直接下載結果到您的電腦**

import time
import os
import sys
import urllib.request
import re
import shutil
from google.colab import files

# ==========================================
#        第一步：環境與變數設定 & 完美破解
# ==========================================
print("🔍 正在初始化環境...")

work_dir = "/content/rf_diffusion_all_atom"
if os.getcwd() != work_dir:
    os.makedirs(work_dir, exist_ok=True)
    os.chdir(work_dir)

sif_filename = "rf_se3_diffusion.sif"
if not os.path.exists(sif_filename):
    print(f"❌ 找不到映像檔，正在下載 (約 2GB)...")
    os.system(f"wget https://files.ipd.uw.edu/pub/rf_diffusion/rf_se3_diffusion.sif")

# ⚡ 安全還原並破解 run_inference.py (預防萬一)
file_path = "run_inference.py"
if os.path.exists(file_path):
    os.system(f"git checkout {file_path} 2>/dev/null")
    with open(file_path, "r") as f:
        code = f.read()
    match = re.search(r'^(\s*)idealize_backbone\.rewrite\(out_unidealized, out_idealized\)', code, re.MULTILINE)
    if match:
        indent = match.group(1)
        safe_new_code = (
            f"{indent}try:\n"
            f"{indent}    idealize_backbone.rewrite(out_unidealized, out_idealized)\n"
            f"{indent}except Exception as e:\n"
            f"{indent}    print('\\n⚠️ [破解成功] 略過骨架微調，強制保留原始結果')\n"
            f"{indent}    import shutil\n"
            f"{indent}    shutil.copy(out_unidealized, out_idealized)"
        )
        code = code.replace(match.group(0), safe_new_code)
        with open(file_path, "w") as f:
            f.write(code)

# ==========================================
#        參數設定區
# ==========================================
# @markdown **1. 輸入 PDB ID 或上傳的 PDB 檔名 (不含副檔名)**
pdb_id = "heme" # @param {type:"string"}
output_prefix = "output/denovo_binder/sample" # @param {type:"string"}

# @markdown **2. 要無中生有生成多長的蛋白質？ (預設 100 個胺基酸)**
contig_settings = "['100-100']" # @param {type:"string"}

# @markdown **3. 擴散模型參數**
diffuser_T = 30 # @param {type:"slider", min:10, max:100, step:5}
num_designs = 1 # @param {type:"integer"}

# 自動下載邏輯
pdb_id = pdb_id.strip().lower()
input_dir = "input"
os.makedirs(input_dir, exist_ok=True)
input_pdb = os.path.join(input_dir, f"{pdb_id}.pdb")
cleaned_pdb = os.path.join(input_dir, f"{pdb_id}_cleaned_binder.pdb")

if not os.path.exists(input_pdb):
    print(f"\n📥 正在從 RCSB PDB 下載 {pdb_id.upper()}...")
    urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb", input_pdb)

# ==========================================
#        第二步：自動解析配體
# ==========================================
print("\n🧬 正在分析 PDB 配體...")
ignore_list = {"HOH", "WAT", "SO4", "PO4", "GOL", "EDO", "ACT", "NAG", "CL", "NA", "K", "MG", "CA", "ZN", "FMT"}
found_ligands = set()

with open(input_pdb, 'r') as f:
    for line in f:
        if line.startswith("HETATM"):
            res_name = line[17:20].strip()
            if res_name not in ignore_list:
                found_ligands.add(res_name)

if not found_ligands:
    raise ValueError(f"❌ 找不到有效配體，請確認 {pdb_id.upper()} 是否包含目標小分子！")

ligand_code = list(found_ligands)[0]
print(f"🎯 鎖定目標配體: 【 {ligand_code} 】")

# ==========================================
#        第三步：清洗 PDB (注入 Dummy 胺基酸)
# ==========================================
print("\n🧹 正在清洗 PDB (注入 Dummy Glycine 並提取配體)...")

# 準備一個假的 Glycine 座標 (Chain Z, 第 999 號)
# 作用：單純為了讓 RFdiffusion 底層程式抓得到 idx_prot，避免報錯。
dummy_glycine = (
    "ATOM      1  N   GLY Z 999       0.000   0.000   0.000  1.00 20.00           N  \n"
    "ATOM      2  CA  GLY Z 999       1.458   0.000   0.000  1.00 20.00           C  \n"
    "ATOM      3  C   GLY Z 999       2.009   1.424   0.000  1.00 20.00           C  \n"
    "ATOM      4  O   GLY Z 999       1.353   2.404   0.000  1.00 20.00           O  \n"
)

with open(input_pdb, 'r') as f_in, open(cleaned_pdb, 'w') as f_out:
    # 1. 寫入假胺基酸騙過 Parser
    f_out.write(dummy_glycine)

    # 2. 提取我們目標的小分子座標
    for line in f_in:
        if line.startswith("HETATM"):
            res_name = line[17:20].strip()
            if res_name == ligand_code:
                # 清除 alternate location 標記，避免重複讀取
                alt_loc = line[16]
                if alt_loc in [' ', 'A']:
                    clean_line = line[:16] + ' ' + line[17:26] + ' ' + line[27:]
                    f_out.write(clean_line)

# ==========================================
#        第四步：執行 RFdiffusion (De novo 結合)
# ==========================================
os.makedirs(os.path.dirname(output_prefix), exist_ok=True)
print(f"\n🚀 開始執行 De novo 結合蛋白設計 (目標: {ligand_code}, 長度: {contig_settings})...")

abs_cleaned_pdb = os.path.abspath(cleaned_pdb)

!singularity run --nv {sif_filename} -u run_inference.py \
  inference.deterministic=True \
  diffuser.T={diffuser_T} \
  inference.output_prefix={output_prefix} \
  inference.input_pdb="{abs_cleaned_pdb}" \
  contigmap.contigs="{contig_settings}" \
  inference.ligand={ligand_code} \
  inference.num_designs={num_designs} \
  inference.design_startnum=0

# ==========================================
#        第五步：自動打包與下載
# ==========================================
expected_pdb = f"{output_prefix}_0.pdb"

if os.path.exists(expected_pdb):
    print(f"\n🎉 執行成功！全新設計的蛋白質已生成。")
    print("📦 正在將結果打包並準備下載...")

    download_dir = "/content/rf_diffusion_all_atom/download_temp"
    os.makedirs(download_dir, exist_ok=True)

    for ext in ['.pdb', '.trb']:
        src_file = f"{output_prefix}_0{ext}"
        if os.path.exists(src_file):
            shutil.copy(src_file, download_dir)

    zip_path = "/content/DeNovo_Binder_Result"
    shutil.make_archive(zip_path, 'zip', download_dir)

    print("✅ 打包完成！瀏覽器即將開始下載...")
    files.download(zip_path + '.zip')
else:
    print(f"\n❌ 執行結束，但未生成檔案，請檢查日誌。")

🔍 正在初始化環境...

🧬 正在分析 PDB 配體...
🎯 鎖定目標配體: 【 HEM 】

🧹 正在清洗 PDB (注入 Dummy Glycine 並提取配體)...

🚀 開始執行 De novo 結合蛋白設計 (目標: HEM, 長度: ['100-100'])...
/bin/sh: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
[05:49:50] /opt/dgl/src/runtime/tensordispatch.cc:43: TensorDispatcher: dlopen failed: /opt/conda/envs/env/lib/python3.9/site-packages/dgl/tensoradapter/pytorch/libtensoradapter_pytorch_1.13.1.so: cannot open shared object file: No such file or directory
Using backend: pytorch
/opt/conda/envs/env/lib/python3.9/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'aa': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
[2026-03-24 05:49:57,020][inference.model_runners][INFO] - Reading checkpoint from RFDiffusionAA_paper_weights.pt
loading RFDiffusionAA_paper_weights.pt
loaded RFDiffusionAA_paper_weights.pt
OVERRIDING: You are changing diffuser.T fro

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>